# 🌳 Tree Import (Simple)

**Import prepared CSV data into the Digital Forest Twin database.**

This notebook is designed for **template-formatted CSVs** that have already been prepared using `prepare_ecosense.ipynb` or similar preprocessing.

## Prerequisites

Your CSV should have columns matching the template format:
- `LocationID` - FK to shared.Locations
- `SpeciesID` - FK to shared.Species  
- `Latitude`, `Longitude` - WGS84 coordinates
- `DBH_cm` - Diameter in centimeters
- `Height_m` - Height in meters (optional)
- `ExternalID` - Unique identifier for linking (optional)

See `data/templates/README.md` for full specification.

## Workflow

1. Connect to database
2. Load prepared CSV
3. Validate data
4. Preview import
5. Execute import (dry-run first)

## Step 1: Setup & Database Connection

In [ ]:
import os
import pandas as pd
import psycopg2
from pathlib import Path
from urllib.parse import quote_plus
from dotenv import load_dotenv

# Load environment from docker/.env
env_path = Path("../docker/.env")
if env_path.exists():
    load_dotenv(env_path)
    print(f"✓ Loaded environment from {env_path}")
else:
    print(f"⚠️  Environment file not found: {env_path}")

# Database connection settings
POSTGRES_HOST = "localhost"
POSTGRES_USER = os.getenv("POSTGRES_USER", "postgres")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_DATABASE = os.getenv("POSTGRES_DB", "postgres")
POSTGRES_PORT = os.getenv("POSTGRES_PORT", "5432")
POOLER_TENANT_ID = os.getenv("POOLER_TENANT_ID", "")

# Supavisor requires tenant ID in username
POSTGRES_USER_POOLER = (
    f"{POSTGRES_USER}.{POOLER_TENANT_ID}" if POOLER_TENANT_ID else POSTGRES_USER
)

if not POSTGRES_PASSWORD:
    raise RuntimeError("❌ POSTGRES_PASSWORD not found in docker/.env")

print(
    f"✓ Database: {POSTGRES_USER_POOLER}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DATABASE}"
)

In [ ]:
# Test database connection
def get_connection():
    return psycopg2.connect(
        host=POSTGRES_HOST,
        user=POSTGRES_USER_POOLER,
        password=POSTGRES_PASSWORD,
        database=POSTGRES_DATABASE,
        port=POSTGRES_PORT,
    )


try:
    conn = get_connection()
    conn.close()
    print("✓ Database connection successful")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("\nTroubleshooting:")
    print("  1. Ensure Docker containers are running: docker compose up -d")
    print("  2. Check docker/.env has correct POSTGRES_PASSWORD")

## Step 2: Load Prepared CSV

In [ ]:
# ============================================================================
# CONFIGURATION - Set your CSV path here
# ============================================================================

CSV_PATH = "../data/ecosense_250911_trees_prepared.csv"

# Alternative paths:
# CSV_PATH = "../data/mathisle_250904_prepared.csv"
# CSV_PATH = "../data/templates/trees_import_template.csv"

# ============================================================================

if not Path(CSV_PATH).exists():
    print(f"❌ File not found: {CSV_PATH}")
    print("\nAvailable files in ../data/:")
    for f in Path("../data").glob("*.csv"):
        print(f"  - {f.name}")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"✓ Loaded {len(df)} rows from {Path(CSV_PATH).name}")
    print(f"\nColumns: {list(df.columns)}")

In [ ]:
# Preview data
print("📋 Data Preview (first 5 rows):")
display(df.head())

## Step 3: Validate Data

In [ ]:
# Required columns for trees.Trees
REQUIRED_COLUMNS = ["LocationID", "SpeciesID", "Latitude", "Longitude"]
OPTIONAL_COLUMNS = [
    "DBH_cm",
    "Height_m",
    "TreeStatusID",
    "FieldNotes",
    "ExternalID",
    "QR_Code_URL",
]

print("📋 Column Validation:")
print("=" * 50)

missing_required = []
for col in REQUIRED_COLUMNS:
    if col in df.columns:
        null_count = df[col].isna().sum()
        if null_count > 0:
            print(f"  ⚠️  {col}: {null_count} NULL values")
        else:
            print(f"  ✓ {col}: OK")
    else:
        print(f"  ❌ {col}: MISSING (required)")
        missing_required.append(col)

print("\n  Optional columns:")
for col in OPTIONAL_COLUMNS:
    if col in df.columns:
        null_count = df[col].isna().sum()
        print(f"  ✓ {col}: {len(df) - null_count}/{len(df)} values")
    else:
        print(f"  - {col}: not present")

if missing_required:
    print(f"\n❌ Cannot import: missing required columns: {missing_required}")
else:
    print(f"\n✓ All required columns present")

In [ ]:
# Validate foreign keys against database
conn = get_connection()

print("🔗 Foreign Key Validation:")
print("=" * 50)

# Check LocationIDs
db_locations = pd.read_sql(
    "SELECT LocationID, LocationName FROM shared.Locations", conn
)
csv_locations = df["LocationID"].dropna().unique()
invalid_locations = [
    loc for loc in csv_locations if loc not in db_locations["LocationID"].values
]

if invalid_locations:
    print(f"  ❌ Invalid LocationIDs: {invalid_locations}")
    print(f"     Valid options: {db_locations['LocationID'].tolist()}")
else:
    print(f"  ✓ LocationIDs: all valid ({len(csv_locations)} unique)")

# Check SpeciesIDs
db_species = pd.read_sql("SELECT SpeciesID, CommonName FROM shared.Species", conn)
csv_species = df["SpeciesID"].dropna().unique()
invalid_species = [sp for sp in csv_species if sp not in db_species["SpeciesID"].values]

if invalid_species:
    print(f"  ❌ Invalid SpeciesIDs: {invalid_species}")
    print(f"     Valid options:")
    for _, row in db_species.iterrows():
        print(f"       {int(row['SpeciesID'])}: {row['CommonName']}")
else:
    print(f"  ✓ SpeciesIDs: all valid ({len(csv_species)} unique)")

conn.close()

# Summary
if invalid_locations or invalid_species:
    print("\n❌ Fix foreign key issues before importing")
else:
    print("\n✓ All foreign keys valid")

In [ ]:
# Validate coordinate ranges
print("📍 Coordinate Validation:")
print("=" * 50)

lat_min, lat_max = df["Latitude"].min(), df["Latitude"].max()
lon_min, lon_max = df["Longitude"].min(), df["Longitude"].max()

print(f"  Latitude:  {lat_min:.6f} to {lat_max:.6f}")
print(f"  Longitude: {lon_min:.6f} to {lon_max:.6f}")

# Check if in Black Forest / Germany region
if 47.0 < lat_min and lat_max < 49.0 and 7.0 < lon_min and lon_max < 9.0:
    print("\n  ✓ Coordinates in expected range (Black Forest region)")
else:
    print("\n  ⚠️  Coordinates may be outside expected area")
    print("     Expected: Lat 47-49°, Lon 7-9° (Black Forest)")

## Step 4: Preview Import

In [ ]:
# Build the import DataFrame
# Map CSV columns to database columns

import_df = pd.DataFrame(
    {
        "LocationID": df["LocationID"].astype(int),
        "SpeciesID": df["SpeciesID"].astype("Int64"),  # Nullable integer
        "VariantTypeID": 1,  # 1 = field_measurement
        "TreeStatusID": df.get("TreeStatusID", 1).fillna(1).astype(int),
        "Height_m": df.get("Height_m"),
        "FieldNotes": df.get("FieldNotes", "").fillna(""),
    }
)

# Add ExternalID to FieldNotes if present
if "ExternalID" in df.columns:
    import_df["FieldNotes"] = (
        import_df["FieldNotes"] + " | ExternalID: " + df["ExternalID"].astype(str)
    )

# Create Position WKT from lat/lon
import_df["Position_WKT"] = df.apply(
    lambda r: (
        f"SRID=4326;POINT({r['Longitude']} {r['Latitude']})"
        if pd.notna(r["Latitude"]) and pd.notna(r["Longitude"])
        else None
    ),
    axis=1,
)

print(f"📋 Import Preview ({len(import_df)} rows):")
print("=" * 60)
display(import_df.head())

print(f"\n📊 Summary:")
print(f"  Rows to insert: {len(import_df)}")
print(f"  With height data: {import_df['Height_m'].notna().sum()}")
print(f"  With position: {import_df['Position_WKT'].notna().sum()}")

## Step 5: Execute Import

In [ ]:
# ============================================================================
# IMPORT CONFIGURATION
# ============================================================================

DRY_RUN = True  # Set to False for actual import

# ============================================================================


def execute_import(import_df: pd.DataFrame, dry_run: bool = True) -> dict:
    """Insert trees into the database."""

    stats = {"attempted": 0, "inserted": 0, "failed": 0, "errors": []}

    conn = get_connection()
    cursor = conn.cursor()

    insert_query = """
        INSERT INTO trees.Trees (
            LocationID, SpeciesID, VariantTypeID, TreeStatusID,
            Height_m, FieldNotes, Position, CreatedBy
        ) VALUES (
            %s, %s, %s, %s, %s, %s, 
            ST_GeomFromEWKT(%s),
            'import_trees_simple.ipynb'
        )
        RETURNING VariantID
    """

    print(
        f"\n{'🔍 DRY RUN' if dry_run else '⚡ LIVE IMPORT'} - {'Testing' if dry_run else 'Inserting'} {len(import_df)} rows"
    )
    print("=" * 60)

    for idx, row in import_df.iterrows():
        stats["attempted"] += 1

        values = (
            row["LocationID"],
            row["SpeciesID"] if pd.notna(row["SpeciesID"]) else None,
            row["VariantTypeID"],
            row["TreeStatusID"],
            row["Height_m"] if pd.notna(row["Height_m"]) else None,
            row["FieldNotes"] if row["FieldNotes"] else None,
            row["Position_WKT"],
        )

        try:
            cursor.execute(insert_query, values)
            stats["inserted"] += 1
        except Exception as e:
            stats["failed"] += 1
            if len(stats["errors"]) < 5:
                stats["errors"].append(f"Row {idx}: {str(e)[:80]}")

    if dry_run:
        conn.rollback()
        print("🔄 Rolled back (dry run)")
    else:
        conn.commit()
        print("✅ Committed to database")

    cursor.close()
    conn.close()

    return stats


# Execute import
stats = execute_import(import_df, dry_run=DRY_RUN)

print(f"\n📊 Import Statistics:")
print(f"  Attempted: {stats['attempted']}")
print(f"  Inserted:  {stats['inserted']}")
print(f"  Failed:    {stats['failed']}")

if stats["errors"]:
    print(f"\n❌ Sample errors:")
    for err in stats["errors"]:
        print(f"  {err}")

if DRY_RUN:
    print("\n" + "=" * 60)
    print("ℹ️  This was a DRY RUN. No data was inserted.")
    print("   Set DRY_RUN = False above and re-run to import.")
    print("=" * 60)

## Step 6: Verify Import

In [ ]:
# Verify imported trees (run after DRY_RUN = False)
conn = get_connection()

query = """
    SELECT 
        t.VariantID,
        l.LocationName,
        s.CommonName as Species,
        t.Height_m,
        ST_Y(t.Position) as Latitude,
        ST_X(t.Position) as Longitude,
        t.CreatedAt,
        t.CreatedBy
    FROM trees.Trees t
    LEFT JOIN shared.Locations l ON t.LocationID = l.LocationID
    LEFT JOIN shared.Species s ON t.SpeciesID = s.SpeciesID
    WHERE t.CreatedBy = 'import_trees_simple.ipynb'
    ORDER BY t.CreatedAt DESC
    LIMIT 10
"""

recent = pd.read_sql(query, conn)
conn.close()

if len(recent) > 0:
    print(f"📋 Recently imported trees ({len(recent)} shown):")
    display(recent)
else:
    print("No trees found with CreatedBy = 'import_trees_simple.ipynb'")
    print("Run import with DRY_RUN = False first.")